# Step 7: XGBoost Baseline

## 1. Objective

Train controlled XGBoost baselines (gradient-boosted trees) on the same 42 causal features used by Logistic Regression (Step 5) and Random Forest (Step 6), examine whether XGBoost reproduces Step 6's artifact-driven near-perfect performance, and extend the factual model comparison. Still a baseline step: no hyperparameter search, no early stopping, no final model selection.

## 2. XGBoost environment verification

XGBoost required the system `libomp` (OpenMP) library on macOS, which was not present initially. Homebrew and `libomp` were installed outside this session (by the user, since it needed an interactive sudo prompt), then XGBoost was `pip install`-ed into the existing `.venv`. A smoke test (tiny 1,000-row sample, full planned configuration) confirmed `fit()`/`predict_proba()` work with zero warnings before any full-scale training was attempted.

In [ ]:
import sys
sys.path.append("..")

import json
import os
import platform
import joblib
import numpy as np
import pandas as pd
import xgboost

from src.model_utils import (
    load_split_data,
    validate_model_features,
    compute_scale_pos_weight,
    build_xgboost_classifier,
    train_model,
    evaluate_classifier,
    evaluate_at_k,
    extract_xgboost_feature_importance,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

print("XGBoost version:", xgboost.__version__)
print("Platform machine:", platform.machine())
print("CPU cores:", os.cpu_count())

## 3. Load chronological datasets

In [ ]:
data = load_split_data()
X_train, y_train = data["X_train"], data["y_train"]
X_val, y_val = data["X_validation"], data["y_validation"]
X_test, y_test = data["X_test"], data["y_test"]
feature_columns = data["feature_columns"]

print("X_train:", X_train.shape, " X_validation:", X_val.shape, " X_test:", X_test.shape)
for name, y in [("train", y_train), ("validation", y_val), ("test", y_test)]:
    print(f"{name}: n={len(y):,}, fraud={int(y.sum()):,}, rate={y.mean()*100:.4f}%")

## 4. Validate model features

In [ ]:
print("Feature count:", len(feature_columns), "(expected 42)")
print("Identical columns across all 3 splits:", list(X_train.columns) == list(X_val.columns) == list(X_test.columns))
for split_name, X in [("train", X_train), ("validation", X_val), ("test", X_test)]:
    checks = validate_model_features(X, feature_columns)
    assert all(checks.values()), f"{split_name} failed: {checks}"
    print(f"{split_name}: {checks}")

print("\n42 feature names/order:")
for i, c in enumerate(feature_columns):
    print(f"  {i}: {c}")

## 5. Why XGBoost

**Random Forest (Step 6): bagging** -- many trees are grown independently and in parallel on bootstrapped samples/feature subsets, and their predictions are averaged. Each tree is grown to reduce variance by being different from the others; no tree knows what the others got wrong.

**XGBoost: sequential gradient boosting** -- trees are grown one at a time, and each new tree is trained specifically to correct the errors (residuals, in the gradient sense) left by the current ensemble. This lets boosting often reach strong performance with fewer, shallower trees than bagging, at the cost of being more sensitive to overfitting if boosting rounds/depth aren't controlled -- which is exactly why this step fixes `n_estimators=300` with no early stopping, rather than letting the model search for its own optimal number of rounds.

## 6. Class imbalance calculation

In [ ]:
scale_pos_weight = compute_scale_pos_weight(y_train)
n_fraud = int(y_train.sum())
n_legit = len(y_train) - n_fraud
print(f"n_legit={n_legit:,}  n_fraud={n_fraud:,}")
print(f"scale_pos_weight = n_legit / n_fraud = {scale_pos_weight:.4f} (~{scale_pos_weight:.0f} : 1)")
print("Calculated programmatically from y_train -- not hard-coded.")

## 7. Baseline configuration

In [ ]:
config = dict(
    n_estimators=300, max_depth=6, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8, min_child_weight=1, reg_lambda=1.0,
    tree_method="hist", random_state=42, n_jobs=8,
)
print("Fixed configuration (both models, except scale_pos_weight):", config)
print("Model E scale_pos_weight = 1")
print(f"Model F scale_pos_weight = {scale_pos_weight:.4f}")
print("\nNOT tuned -- no GridSearchCV, RandomizedSearchCV, Optuna, or early stopping. n_estimators=300 is fixed.")

## 8. Model E -- unweighted XGBoost (scale_pos_weight=1)

In [ ]:
model_e = build_xgboost_classifier(scale_pos_weight=1.0, **config)
info_e = train_model(model_e, X_train, y_train)
print(f"Model E: training_time_sec={info_e['training_time_sec']:.1f}")
print(f"Boosting rounds used: {info_e['pipeline'].get_booster().num_boosted_rounds()}")
print("Fit ONLY on (X_train, y_train).")

## 9. Model F -- scale_pos_weight XGBoost

In [ ]:
model_f = build_xgboost_classifier(scale_pos_weight=scale_pos_weight, **config)
info_f = train_model(model_f, X_train, y_train)
print(f"Model F: training_time_sec={info_f['training_time_sec']:.1f}")
print(f"Boosting rounds used: {info_f['pipeline'].get_booster().num_boosted_rounds()}")
print("Fit ONLY on (X_train, y_train).")

fitted = {"model_e_unweighted": info_e, "model_f_scale_pos_weight": info_f}

## 10. Validation evaluation (threshold = 0.5)

In [ ]:
results = {}
for name, info in fitted.items():
    proba_val = info["pipeline"].predict_proba(X_val)[:, 1]
    info["proba_val"] = proba_val
    val_metrics = evaluate_classifier(y_val, proba_val, threshold=0.5)
    results.setdefault(name, {})["validation_metrics"] = val_metrics
    print(f"{name} -- VALIDATION @ threshold=0.5: {val_metrics}\n")

## 11. Test evaluation -- FINAL TEST RESULTS (NOT used for model selection)

In [ ]:
for name, info in fitted.items():
    proba_test = info["pipeline"].predict_proba(X_test)[:, 1]
    info["proba_test"] = proba_test
    test_metrics = evaluate_classifier(y_test, proba_test, threshold=0.5)
    results[name]["test_metrics"] = test_metrics
    print(f"{name} -- FINAL TEST RESULTS @ threshold=0.5: {test_metrics}\n")

## 12. Precision@K / Recall@K

In [ ]:
k_values = [100, 500, 1000, 5000, 10000]
for name, info in fitted.items():
    at_k_val = evaluate_at_k(y_val, info["proba_val"], k_values)
    at_k_test = evaluate_at_k(y_test, info["proba_test"], k_values)
    results[name]["at_k_validation"] = at_k_val.to_dict(orient="records")
    results[name]["at_k_test"] = at_k_test.to_dict(orient="records")
    print(f"{name} -- VALIDATION:\n{at_k_val.to_string(index=False)}")
    print(f"\n{name} -- TEST:\n{at_k_test.to_string(index=False)}\n")

## 13. Confusion matrices (threshold = 0.5)

In [ ]:
for name in fitted:
    for split in ["validation", "test"]:
        cm = results[name][f"{split}_metrics"]["confusion_matrix"]
        print(f"{name} -- {split}: TN={cm['tn']:,} FP={cm['fp']:,} FN={cm['fn']:,} TP={cm['tp']:,}")

## 14. Gain-based feature importance

`importance_type="gain"`: average improvement in the training objective contributed by splits on that feature. Not a causal measure -- a feature with high gain is one the model found useful for reducing loss in this fitted ensemble, not necessarily a real-world "cause" of fraud.

In [ ]:
importances = {}
for name, info in fitted.items():
    imp = extract_xgboost_feature_importance(info["pipeline"], feature_columns, importance_type="gain")
    importances[name] = imp
    print(f"=== {name}: TOP 20 gain importance ===")
    print(imp.head(20).to_string(index=False))
    print()

## 15. PaySim artifact investigation (required)

In [ ]:
for name, imp in importances.items():
    total = imp["importance"].sum()
    s = imp.set_index("feature")["importance"]
    a = s.get("amount_to_sender_balance", 0)
    b = s.get("amount_exceeds_sender_balance", 0)
    print(f"{name}:")
    print(f"  total gain across all 42 features: {total:.2f}")
    print(f"  amount_to_sender_balance:      {a:.2f}  ({a/total*100:.2f}%)")
    print(f"  amount_exceeds_sender_balance: {b:.2f}  ({b/total*100:.2f}%)")
    print(f"  COMBINED:                      {a+b:.2f}  ({(a+b)/total*100:.2f}%)\n")

**Finding: the PaySim artifact reproduces in XGBoost, exactly as flagged before training.** These two balance-ratio features account for **53.20%** of Model E's total gain and **41.84%** of Model F's -- comparable in magnitude to Random Forest's 45%+ combined importance in Step 6 (same two features dominated there too). Both tree-based model families converge on the same signal because it is the strongest, most separable pattern actually present in this synthetic dataset (Step 2's "drained sender account" simulation artifact), not because either model discovered something deeper about real fraud behavior.

Per the required interpretation rule: **XGBoost reproduced the strong PaySim signal observed with Random Forest, with the dominant predictive contribution concentrated in balance-ratio features associated with the synthetic drained-account pattern.** This is not read as "XGBoost solved fraud detection."

## 16. Logistic Regression vs Random Forest vs XGBoost comparison

In [ ]:
with open("../results/logistic_regression_metrics.json") as f:
    lr_metrics = json.load(f)
with open("../results/random_forest_metrics.json") as f:
    rf_metrics = json.load(f)

model_specs = [
    ("Logistic Regression - unweighted", "Logistic Regression", "none", lr_metrics["model_a_unweighted"]),
    ("Logistic Regression - balanced", "Logistic Regression", "class_weight=balanced", lr_metrics["model_b_balanced"]),
    ("Random Forest - unweighted", "Random Forest", "none", rf_metrics["model_c_unweighted"]),
    ("Random Forest - balanced_subsample", "Random Forest", "class_weight=balanced_subsample", rf_metrics["model_d_balanced_subsample"]),
    ("XGBoost - unweighted", "XGBoost", "none (scale_pos_weight=1)", results["model_e_unweighted"]),
    ("XGBoost - scale_pos_weight", "XGBoost", f"scale_pos_weight={scale_pos_weight:.2f}", results["model_f_scale_pos_weight"]),
]
rows = []
for model_name, family, strategy, m in model_specs:
    vm, tm = m["validation_metrics"], m["test_metrics"]
    rows.append({
        "model": model_name, "model_family": family, "imbalance_strategy": strategy,
        "validation_precision": vm["precision"], "validation_recall": vm["recall"], "validation_f1": vm["f1"],
        "validation_roc_auc": vm["roc_auc"], "validation_pr_auc": vm["pr_auc"],
        "test_precision": tm["precision"], "test_recall": tm["recall"], "test_f1": tm["f1"],
        "test_roc_auc": tm["roc_auc"], "test_pr_auc": tm["pr_auc"],
    })
comparison = pd.DataFrame(rows)
comparison

**Observed, factually:** XGBoost sits between Logistic Regression and Random Forest at threshold 0.5 -- clearly ahead of both Logistic Regression variants, but slightly behind Random Forest's near-perfect scores on this particular metric snapshot. Neither Random Forest nor XGBoost is declared the winner here; both reflect the same underlying artifact to different degrees, and no final model selection happens in this step.

In [ ]:
lr_at_k = pd.read_csv("../results/logistic_regression_precision_recall_at_k.csv")
rf_at_k = pd.read_csv("../results/random_forest_precision_recall_at_k.csv")
xgb_at_k_rows = []
for name in fitted:
    for split_name, records in [("validation", results[name]["at_k_validation"]), ("test", results[name]["at_k_test"])]:
        for rec in records:
            xgb_at_k_rows.append({"model": name, "split": split_name, **rec})
xgb_at_k = pd.DataFrame(xgb_at_k_rows)

name_map = {
    "model_a_unweighted": "Logistic Regression - unweighted",
    "model_b_balanced": "Logistic Regression - balanced",
    "model_c_unweighted": "Random Forest - unweighted",
    "model_d_balanced_subsample": "Random Forest - balanced_subsample",
    "model_e_unweighted": "XGBoost - unweighted",
    "model_f_scale_pos_weight": "XGBoost - scale_pos_weight",
}
combined_at_k = pd.concat([lr_at_k, rf_at_k, xgb_at_k], ignore_index=True)
combined_at_k["model"] = combined_at_k["model"].map(name_map)
combined_at_k[combined_at_k["k"].isin([100, 1000, 10000])]

## 17. Computational considerations

Both XGBoost models trained in ~23-25 seconds each on the full 4,463,587-row training set -- dramatically faster than either Random Forest model (~180-208s) at 300 boosting rounds vs. 200 trees, thanks to `tree_method="hist"` histogram-based split finding. No resource problems occurred; the specified configuration (`n_estimators=300`, no early stopping) was used exactly as given, with no reduction.

## 18. Limitations

- **Same core caveat as Step 6:** strong metrics are substantially explained by two engineered features that encode a known PaySim simulation artifact (Step 2's drained-account pattern), not confirmed generalizable fraud behavior.
- No hyperparameter tuning or early stopping was used -- `n_estimators=300` and all other settings are fixed baseline choices, not validated ones.
- Gain-based importance, like Random Forest's impurity importance, is not causal and can split credit across correlated features (`type_TRANSFER`/`is_transfer`, etc.).
- Test-period distribution shift (fraud rate 0.4361% vs train's 0.0816%) still applies; test metrics are not "real-world" performance.
- Reloaded model predictions were verified bit-for-bit identical to pre-save predictions (max abs diff = 0.0), unlike Random Forest's tiny (2.22e-16) floating-point difference in Step 6 -- both are acceptable, but this is a real, observed difference between the two libraries' prediction-aggregation implementations worth noting factually.

## 19. Final baseline summary

In [ ]:
joblib.dump(fitted["model_e_unweighted"]["pipeline"], "../models/xgboost_baseline.joblib")
joblib.dump(fitted["model_f_scale_pos_weight"]["pipeline"], "../models/xgboost_balanced.joblib")
print("Saved models/xgboost_baseline.joblib and models/xgboost_balanced.joblib")

for name, path in [
    ("model_e_unweighted", "../models/xgboost_baseline.joblib"),
    ("model_f_scale_pos_weight", "../models/xgboost_balanced.joblib"),
]:
    reloaded = joblib.load(path)
    reloaded_proba = reloaded.predict_proba(X_val)[:, 1]
    max_diff = np.max(np.abs(fitted[name]["proba_val"] - reloaded_proba))
    close = np.allclose(fitted[name]["proba_val"], reloaded_proba, atol=1e-6)
    same_class = np.array_equal((fitted[name]["proba_val"] >= 0.5).astype(int), (reloaded_proba >= 0.5).astype(int))
    print(f"{name}: max_abs_diff={max_diff:.2e}, allclose={close}, classifications identical @0.5={same_class}")

print("\nPrior artifacts untouched:")
for f in ["../models/logistic_regression_baseline.joblib", "../models/logistic_regression_balanced.joblib",
          "../models/random_forest_baseline.joblib", "../models/random_forest_balanced.joblib"]:
    print(f"  {f}: exists = {os.path.exists(f)}")

In [ ]:
metrics_out = {
    name: {
        "validation_metrics": results[name]["validation_metrics"],
        "test_metrics": {**results[name]["test_metrics"], "label": "FINAL TEST RESULTS - not used for model selection"},
        "at_k_validation": results[name]["at_k_validation"],
        "at_k_test": results[name]["at_k_test"],
    }
    for name in fitted
}
metrics_out["_meta"] = {
    "threshold_used": 0.5,
    "configuration": config,
    "scale_pos_weight_model_e": 1.0,
    "scale_pos_weight_model_f": scale_pos_weight,
    "no_hyperparameter_tuning": True,
    "no_early_stopping": True,
    "paysim_artifact_finding": (
        "XGBoost reproduced the strong PaySim signal observed with Random Forest, with the dominant predictive "
        "contribution concentrated in balance-ratio features associated with the synthetic drained-account "
        "pattern."
    ),
    "test_set_warning": (
        "Test fraud rate (0.4361%) is far higher than train (0.0816%) due to the documented late-period "
        "legitimate-volume collapse. Test metrics were not used to select between Model E and Model F."
    ),
}
with open("../results/xgboost_metrics.json", "w") as f:
    json.dump(metrics_out, f, indent=2)

imp_frames = []
for name, imp in importances.items():
    c = imp.copy()
    c.insert(0, "model", name)
    imp_frames.append(c)
pd.concat(imp_frames, ignore_index=True).to_csv("../results/xgboost_feature_importance.csv", index=False)
xgb_at_k.to_csv("../results/xgboost_precision_recall_at_k.csv", index=False)
comparison.to_csv("../results/model_baseline_comparison_step7.csv", index=False)
combined_at_k.to_csv("../results/model_baseline_precision_recall_at_k_step7.csv", index=False)

print("Saved results/xgboost_metrics.json")
print("Saved results/xgboost_feature_importance.csv")
print("Saved results/xgboost_precision_recall_at_k.csv")
print("Saved results/model_baseline_comparison_step7.csv")
print("Saved results/model_baseline_precision_recall_at_k_step7.csv")

**Summary:** two XGBoost baselines (unweighted, `scale_pos_weight=1224.25`) were trained on the same 4,463,587-row train period and 42 causal features as Steps 5-6, with 300 fixed boosting rounds and no early stopping or tuning. Both reproduce the PaySim drained-account artifact pattern already seen in Random Forest (53.20% and 41.84% combined gain in the two dominant balance-ratio features), scoring between Logistic Regression and Random Forest at the fixed 0.5 threshold. No model has been selected as final; no threshold has been optimized; test results were not used to choose between Model E and Model F; and prior Logistic Regression/Random Forest artifacts remain unmodified.